# WiseBlockForge Generated Notebook
This notebook was automatically compiled from a visual graph pipeline.


In [ ]:
# Node: data_input_1 (data_input)
print('##NODE_START:data_input_1')
import torch
from torchvision import datasets, transforms
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
print('Loading dataset: MNIST...')
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)
print('Dataset MNIST loaded successfully.')
print('##NODE_COMPLETE:data_input_1')


In [ ]:
# Node: model_training_1 (model_training)
print('##NODE_START:model_training_1')
import torch.nn as nn
import torch.optim as optim

class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleMLP()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

epochs = 1
print(f'Training model on {device} for {epochs} epoch(s)...')
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (data_batch, target_batch) in enumerate(train_loader):
        data_batch, target_batch = data_batch.to(device), target_batch.to(device)
        optimizer.zero_grad()
        output = model(data_batch)
        loss = criterion(output, target_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 200 == 199:
            print(f'Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {running_loss/200:.4f}')
            running_loss = 0.0
print('Model training completed.')
print('##NODE_COMPLETE:model_training_1')


In [ ]:
# Node: evaluation_1 (evaluation)
print('##NODE_START:evaluation_1')
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data_batch, target_batch in test_loader:
        data_batch, target_batch = data_batch.to(device), target_batch.to(device)
        output = model(data_batch)
        test_loss += criterion(output, target_batch).item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target_batch.view_as(pred)).sum().item()
test_loss /= len(test_loader.dataset)
accuracy = 100. * correct / len(test_loader.dataset)
print(f'Test set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)')
print('##NODE_COMPLETE:evaluation_1')
